# Quantia — Measurement Baseline V1
Revisión offline de evidencia agregada. No conecta a bases ni ejecuta migraciones.
Fuente: `evidence/measurement-baseline-v1.json`. Los totales de filas no son trades ni rentabilidad.

In [ ]:
import json
from pathlib import Path
candidates = [Path("docs/audits/evidence/measurement-baseline-v1.json"), Path("evidence/measurement-baseline-v1.json")]
source = next(p for p in candidates if p.exists())
evidence = json.loads(source.read_text(encoding="utf-8"))
tables = evidence["backup_vs_local"]["tables"]
assert evidence["original_qeef_exact_replay_match"]
print("Base main:", evidence["base_main_sha"])
print("Corte local:", evidence["backup_vs_local"]["live_meta"]["source_as_of"])

In [ ]:
print("Tabla | Backup | Local | Ausentes local | Contenido distinto")
for table, r in sorted(tables.items()):
    print(table, r["backup"]["n"], r["local"]["n"], r["backup_missing_local"], r["changed_content"], sep=" | ")
assert sum(r["backup_missing_local"] for r in tables.values()) == 5
assert all(r["changed_scope"] == 0 for r in tables.values())
assert all(not d["groups"] for r in tables.values() for d in r["duplicate_keys"].values())

## Cuenta y límites
La prueba de valuación compara `sum(quantity * current_price)` con el valor invertido informado, a 0,01 ARS. No interpreta el residual como pérdida.
La implementación con Decimal, tratamiento de faltantes y FX está en `scripts/audit_measurement_offline.py`; las consultas exactas están en `scripts/measurement_checks.sql`.

In [ ]:
quality = evidence["quality_checks"]["checks"]
for check in ["fill_decision_owner_classes", "hold_parent_integrity", "movement_fill_currency_identity"]:
    print(check, json.dumps(quality[check], ensure_ascii=False))
print("Valuación:", evidence["offline"]["current_valuation"])
for key in ["qeef_original", "qeef_current"]:
    result = evidence[key]
    print(key, "casos", result["denominator"], "fills", result["fills"], "PnL", result["economic_pnl_net"])
    assert result["economic_pnl_net"] is None

In [ ]:
for name, cohorts in evidence["changed_fields"]["tables"].items():
    for cohort, changes in cohorts.items():
        assert changes["later_version_conflicts"] == 0
        print(name, cohort, changes["rows"], changes["changed_fields"])

## Reproducción
`MEASUREMENT_BASELINE_V1.md` documenta las capturas, fuentes, hashes y comandos de solo lectura. Las pruebas usan fixtures sintéticos.
Una ejecución nueva contra datos vivos representa otro corte; no debe sobrescribir esta publicación. Los datos privados no se incluyen en el repositorio.